# Compare association evidence across breeds

The Labrador Retriever and German Shorthaired Pointer scans remain separate analyses. Here we ask whether the same measured allele has consistent association evidence across the two breeds. We start with an inverse-variance weighted meta-analysis of GEMMA effects, then show GMMAT’s separate score meta-analysis. Fisher’s method and an exploratory pooled scan follow.

GEMMA remains the laboratory reporting workflow. GMMAT is a methodological comparison, and results from the two methods are never combined. These analyses use measured genotypes; reference-panel imputation is deferred. Meta-analysis preserves the breed-specific model fits, but it inherits their small-case and variance-boundary limitations. Ordering the methods does not make any result a confirmed discovery.

## Match the measured variants

We match shared markers by identifier and require identical CanFam4 chromosome, position, REF and ALT alleles. A variant must meet the 2.5% minor-allele frequency requirement separately in each breed to enter the combined analysis. The dogs have disjoint identifiers; statistical independence between breeds is assumed, rather than established by those identifiers alone.

Scheme A is primary. Scheme B changes the control definition and remains a sensitivity comparison with the same cases. Each analysis uses a Bonferroni cutoff of 0.05 divided by the shared aligned candidate count before additional cohort filtering. We do not correct across methods or test types, so this is not a family-wide cutoff for all results in the chapter.

In [ ]:
from pathlib import Path
import sys
study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))
from notebook_support import read_json, run_script, file_digest
from IPython.display import Markdown, Image, display
import os, subprocess
meta_folder = study_root / "results/private/version2/08_cross_breed"
pooled_run = study_root / "data/gwas/pooled-measured-20261004"
pooled_folder = meta_folder / "pooled"
configuration = read_json(study_root / "config/cross_breed.json")

# Set True to rebuild summaries from the saved breed-specific fits.
# This verifies native output receipts and can take several minutes.
rebuild_meta = False
if rebuild_meta or not (meta_folder / "summary.private.json").exists():
    run_script(study_root, "cross_breed_meta.py", "--results-dir", meta_folder)
if rebuild_meta or not (meta_folder / "verification.private.json").exists():
    if os.name == "nt":
        raise RuntimeError("Use the WSL/Linux project kernel for native R verification.")
    association_r = Path.home() / "anaconda3/envs/tsa2024-association/bin/Rscript"
    subprocess.run([str(association_r), str(study_root / "scripts/check_cross_breed.R"), str(study_root)], check=True)
meta = read_json(meta_folder / "summary.private.json")
assert meta["status"] == "cross_breed_meta_complete"
assert meta["config_sha256"] == file_digest(study_root / "config/cross_breed.json")
assert meta["script_sha256"] == file_digest(study_root / "scripts/cross_breed_meta.py")
assert meta["original_sources_unchanged"]
lines = ["| Comparison | Shared candidates | Tested in both breeds | Cutoff | Unavailable effect pairs |",
         "| --- | ---: | ---: | ---: | ---: |"]
for r in meta["families"]:
    lines.append(f"| {r['scenario']} | {r['common_candidates']:,} | {r['eligible_both']:,} | {r['threshold']:.3g} | {r['invalid_effect_pairs']} |")
display(Markdown("\n".join(lines)))

## Combine GEMMA effect estimates

For each shared marker, the inverse-variance analysis combines the two ALT effects, weighting each by the inverse of its squared standard error. A more precise estimate receives more weight. The combined estimate assumes a common underlying effect on GEMMA’s binary-outcome scale; it is not a log odds ratio. Its probability uses a normal approximation, rather than combining the original GEMMA Wald probabilities.

We retain both breed effects, their weights and approximate 95% intervals. Cochran’s Q and I² describe disagreement between the effects. With only two breeds, these measures are imprecise: a nonsignificant Q does not establish that the effects agree. Invalid or nonpositive standard errors exclude the effect pair, without excluding that marker from other valid tests.

The case fractions differ between breeds. Effects from a linear model of an ascertained binary outcome need not represent the same underlying disease-risk effect across those sampling fractions. We combine the numerical effects as a sensitivity comparison and do not interpret them as pooled odds ratios or population risks. No automatic genomic-control rescaling is applied.

In [ ]:
import numpy as np
from scipy.stats import norm

def inverse_variance_effect(effects, standard_errors):
    """Combine effects expressed for the same allele on the same scale."""
    effects = np.asarray(effects, dtype=float)
    standard_errors = np.asarray(standard_errors, dtype=float)
    if not np.isfinite(effects).all() or not np.isfinite(standard_errors).all() or (standard_errors <= 0).any():
        raise ValueError("Both effects need finite, positive standard errors.")
    weights = 1 / standard_errors**2
    combined = np.sum(weights * effects) / weights.sum()
    error = np.sqrt(1 / weights.sum())
    probability = 2 * norm.sf(abs(combined / error))
    return combined, error, probability

def show_analysis_rows(records):
    lines = ["| Comparison | Analysis | Valid tests | Below cutoff | Minimum p |",
             "| --- | --- | ---: | ---: | ---: |"]
    for r in records:
        lines.append(f"| {r['scenario']} | {r['analysis']} | {r['valid_tests']:,} | {r['below_cutoff']} | {r['minimum_p']:.3g} |")
    display(Markdown("\n".join(lines)))

show_analysis_rows([r for r in meta["analyses"] if r["p_column"] == "iv_p"])
for family in meta["families"]:
    display(Markdown("### " + family["scenario"] + ": effect estimates"))
    display(Image(filename=str(meta_folder / (family["scenario"] + "_effects.png"))))

In [ ]:
import csv, gzip
with gzip.open(meta_folder / (configuration["primary_scheme"] + ".private.tsv.gz"), "rt", encoding="utf-8") as handle:
    primary_meta = list(csv.DictReader(handle, delimiter="\t"))
top_effects = sorted((r for r in primary_meta if np.isfinite(float(r["iv_p"]))), key=lambda r: float(r["iv_p"]))[:10]
lines = ["| Marker | ALT | LR effect | GSP effect | LR weight | Q p | I² | Same direction |",
         "| --- | --- | ---: | ---: | ---: | ---: | ---: | --- |"]
for r in top_effects:
    lines.append(f"| {r['marker']} | {r['ALT']} | {float(r['LR_GEMMA_beta']):.3g} | {float(r['GSP_GEMMA_beta']):.3g} | {float(r['LR_weight_fraction']):.1%} | {float(r['Q_p']):.3g} | {float(r['I2']):.1%} | {r['same_effect_direction']} |")
display(Markdown("\n".join(lines)))

## Compare the separate GMMAT score evidence

The saved GMMAT scans contain an aligned score and its information variance for each breed, rather than fitted per-marker effects and standard errors. We sum scores and variances across breeds and test the squared summed score divided by its summed variance against a one-degree-of-freedom chi-square distribution. This is a score meta-analysis, not an inverse-variance analysis of fitted logistic effects.

We check the result against GMMAT’s native `glmm.score.meta` function. GMMAT scores are combined only with GMMAT scores. Opposing directions can cancel, which is appropriate for this common-direction comparison.

In [ ]:
show_analysis_rows([r for r in meta["analyses"] if r["p_column"] == "score_meta_p"])
# Native verification also independently checks all GEMMA effect and Fisher calculations in R.
verification = read_json(meta_folder / "verification.private.json")
assert verification["status"] == "all_cross_breed_meta_calculations_verified"
verification["checks"]

## Explore Fisher’s combination of probabilities

Fisher’s method asks whether there is association evidence across the breeds without requiring matching effect directions. We calculate −2 × the sum of the two log probabilities and compare it with a four-degree-of-freedom chi-square distribution. A strong combined result can be driven by one breed or by opposing effects.

GEMMA Wald, likelihood-ratio and score probabilities are combined separately across breeds. GMMAT has its own Fisher combination. We never combine the two methods or combine different GEMMA tests with one another. These are exploratory comparisons, not independent confirmations of the inverse-variance analysis.

In [ ]:
from scipy.stats import chi2

def fishers_probability(probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    if not np.isfinite(probabilities).all() or ((probabilities <= 0) | (probabilities > 1)).any():
        raise ValueError("Fisher's method needs finite probabilities in (0, 1].")
    return chi2.sf(-2 * np.log(probabilities).sum(), 2 * len(probabilities))

show_analysis_rows([r for r in meta["analyses"] if r["p_column"].endswith("_fisher")])
for family in meta["families"]:
    display(Markdown("### " + family["scenario"] + ": combined evidence"))
    for suffix in ["_manhattan.png", "_qq.png"]:
        display(Image(filename=str(meta_folder / (family["scenario"] + suffix))))

## Explore a pooled measured-genotype model

This final analysis is a sensitivity exercise. Its inclusion in the publication remains undecided. Combining the dogs requires a new model; the within-breed structure checks alone do not establish its suitability.

We use the same shared measured-marker family, require the frequency cutoff in each breed, and adjust for breed and sex. Missing numerical inputs receive the mean ALT count of their own breed and comparison group. Original measurements remain unchanged. GEMMA and GMMAT receive identical numerical genotypes and the same chromosome-excluded genetic relationship matrix.

For the relationship matrix, shared background markers come from the union of the two existing LD-pruned sets. Genotypes are centered and scaled within each broader breed before the joint cross-product. This measures residual genetic similarity after breed means are removed, including cross-breed entries; it neither concatenates the old matrices nor assumes a known pedigree. Union markers are not newly pruned in the pooled sample. The models assume one genetic variance component, which may inadequately represent breed differences. No additional PCs are included. These choices are exploratory and are not validated by convergence or a satisfactory-looking QQ plot.

In [ ]:
# Fresh inference requires the WSL/Linux project kernel and native association tools.
# Progress is printed by comparison group and chromosome. Completed jobs are verified and reused.
run_pooled_models = False
if run_pooled_models:
    run_script(study_root, "pooled_gwas.py", "--output", pooled_run, linux=True)
    run_script(study_root, "check_pooled_inputs.py", linux=True)
    association_r = Path.home() / "anaconda3/envs/tsa2024-association/bin/Rscript"
    subprocess.run([str(association_r), str(study_root / "scripts/check_pooled_scores.R"), str(study_root)], check=True)

# Rebuild figure summaries after completing inference; this does not refit models.
rebuild_pooled_summary = False
if run_pooled_models or rebuild_pooled_summary or not (pooled_folder / "summary.private.json").exists():
    run_script(study_root, "summarize_pooled.py", "--run-dir", pooled_run, "--results-dir", pooled_folder)
pooled = read_json(pooled_folder / "summary.private.json")
assert pooled["status"] == "exploratory_pooled_review_complete"
assert pooled["original_sources_unchanged"]
assert pooled["config_sha256"] == file_digest(study_root / "config/cross_breed.json")
assert read_json(pooled_run / "input-check.private.json")["status"] == "pooled_relationships_and_selected_inputs_verified"
assert read_json(pooled_run / "score-check.private.json")["status"] == "pooled_sample_orders_covariates_and_selected_scores_verified"
show_analysis_rows(pooled["analyses"])
lines = ["| Comparison | Converged GMMAT chromosome nulls | GEMMA ML variance-boundary markers | Median χ² ratio, Wald / GMMAT score |",
         "| --- | ---: | ---: | --- |"]
for group in pooled["groups"]:
    scheme = group["scenario"]
    nulls = sum(r["gmmat_null_converged"] for r in pooled["diagnostics"] if r["scenario"] == scheme)
    wald = next(r for r in pooled["analyses"] if r["scenario"] == scheme and r["analysis"] == "GEMMA Wald")
    score = next(r for r in pooled["analyses"] if r["scenario"] == scheme and r["analysis"] == "GMMAT score")
    lines.append(f"| {scheme} | {nulls}/38 | {wald['gemma_mle_boundary_count']:,} | {wald['median_chi_square_ratio']:.3f} / {score['median_chi_square_ratio']:.3f} |")
display(Markdown("\n".join(lines)))
for group in pooled["groups"]:
    display(Markdown(f"### {group['scenario']}: {group['dogs']} dogs, {group['cases']} cases"))
    for suffix in ["_manhattan.png", "_qq.png"]:
        display(Image(filename=str(pooled_folder / (group["scenario"] + suffix))))

## Read the comparisons together

Agreement in allele direction and a combined effect can support a shared pattern, but neither meta-analysis nor pooling repairs uncertain breed-specific fits. Fisher’s results answer a broader question and can remain strong when a shared-direction effect is weak. Scheme B uses many of the same dogs and is not a replication cohort. All probability approximations remain sensitive to our small case counts, relatedness assumptions and sparse genotype groups.

The full tables retain breed effects, uncertainty, frequencies, sample sizes, heterogeneity and separate probabilities. The cross-breed analyses do not replace the historical Labrador replication chapter or establish that any marker is causal. Technical details are in [08 Cross-breed association](../docs/methods/08%20Cross-breed%20association.md).